<a href="https://colab.research.google.com/github/GuardinTheDev/Is-This-Text-Ai-/blob/Toxenizer/Tokenizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers

In [3]:
import torch
import math
import re
import statistics
from transformers import AutoTokenizer, AutoModelForCausalLM
from collections import Counter

# Ekran kartında çalıştırma yoksa cpuda çalıştırma
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[DONANIM] İşlemler {device.upper()} üzerinde gerçekleştirilecek.\n")

def setup_tokenizer(model_name="distilbert-base-uncased"):
    """
    Belirtilen model için tokenizer'ı yükler.
    DistilBERT, hızlı çalışması ve tespit modellerinde sıkça
    tercih edilmesi nedeniyle varsayılan olarak ayarlanmıştır.
    """
    print(f"[{model_name}] tokenizer'ı yükleniyor...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    return tokenizer

def tokenize_input_text(tokenizer, text):
    """
    Girilen metni yapay zekanın anlayacağı token parçacıklarına
    ve karşılık gelen tensör ID'lerine dönüştürür.
    """
    # Metni token'lara ayır
    tokens = tokenizer.tokenize(text)

    # Modelin matematiksel olarak işleyebilmesi için token'ları ID'lere çevir
    token_ids = tokenizer.convert_tokens_to_ids(tokens)

    return tokens, token_ids

def calculate_perplexity(text, model_name="gpt2"):
    """
    Metnin Perplexity (Şaşkınlık) değerini hesaplar.
    Düşük değer: AI üretimi olma ihtimali yüksek.
    Yüksek değer: İnsan yazımı olma ihtimali yüksek.
    """
    print(f"\n[{model_name}] modeli ile Perplexity hesaplanıyor...")

    ppl_tokenizer = AutoTokenizer.from_pretrained(model_name)
    ppl_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    encodings = ppl_tokenizer(text, return_tensors="pt")
    input_ids = encodings.input_ids.to(device)

    with torch.no_grad():
        outputs = ppl_model(input_ids, labels=input_ids)
        loss = outputs.loss

    perplexity_score = torch.exp(loss).item()

    del ppl_model
    torch.cuda.empty_cache()

    return perplexity_score

def calculate_burstiness(text):
    """
    Cümle uzunluklarının standart sapmasını (Burstiness) hesaplar.
    Düşük sapma: Robotik/AI metin. Yüksek sapma: Doğal/İnsan metni.
    """
    # Metni cümlelere ayır (. ! ? işaretlerinden)
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if len(s.strip()) > 0]

    if len(sentences) < 2:
        return 0.0 # Karşılaştıracak yeterli cümle yoksa sapma sıfırdır

    # Her cümlenin kaç kelimeden oluştuğunu bul
    sentence_lengths = [len(s.split()) for s in sentences]

    # Standart sapmayı hesapla
    burstiness_score = statistics.stdev(sentence_lengths)
    return burstiness_score

def calculate_flesch_kincaid(text):
    """
    Metnin Flesch-Kincaid okunabilirlik (eğitim seviyesi) skorunu hesaplar.
    """
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if len(s.strip()) > 0]
    words = text.split()

    if len(sentences) == 0 or len(words) == 0:
        return 0.0

    total_sentences = len(sentences)
    total_words = len(words)

    # Basit bir İngilizce hece sayıcı (Sesli harfleri sayar)
    vowels = "aeiouyAEIOUY"
    total_syllables = 0
    for word in words:
        count = 0
        if word[0] in vowels:
            count += 1
        for index in range(1, len(word)):
            if word[index] in vowels and word[index - 1] not in vowels:
                count += 1
        if word.endswith("e"):
            count -= 1
        if count == 0:
            count += 1
        total_syllables += count

    # Flesch Okunabilirlik Formülü
    fk_score = 206.835 - 1.015 * (total_words / total_sentences) - 84.6 * (total_syllables / total_words)
    return fk_score



[DONANIM] İşlemler CPU üzerinde gerçekleştirilecek.



In [4]:
# Test Bloğu
if __name__ == "__main__":
    # Örnek bir metin (Yapay zeka üretimi veya insan yazımı olabilir)
    sample_text = "Quantum computing represents a fundamental shift in how we process information. Unlike classical computers that rely on binary bits, quantum systems utilize qubits, which can exist in multiple states simultaneously due to the principles of superposition and entanglement. This unique characteristic allows quantum computers to solve complex mathematical problems at unprecedented speeds. Furthermore, industries ranging from cryptography to pharmaceuticals are investing heavily in this emerging technology. While significant hardware challenges remain, the potential applications of quantum algorithms are vast and transformative for the future of computational science."

    # 1. Modülü başlat
    my_tokenizer = setup_tokenizer()

    # 2. Metni tokenize et
    extracted_tokens, extracted_ids = tokenize_input_text(my_tokenizer, sample_text)

    # 3. Sonuçları raporda sunmak için ekrana yazdır
    print("\n--- Orijinal Metin ---")
    print(sample_text)

    print("\n--- Çıkarılan Tokenlar ---")
    print(extracted_tokens)

    print("\n--- Modelin Göreceği Sayısal ID'ler ---")
    print(extracted_ids)

    print(f"\nToplam Token Sayısı: {len(extracted_tokens)}")

    token_frekanslari = Counter(extracted_tokens)

    print("\n" + "="*40)
    print(f"| {'Token (Sub-word)':<20} | {'Frekans (Adet)':<12} |")
    print("="*40)

    # Perplexity (Şaşkınlık) Hesaplaması
    ppl_skoru = calculate_perplexity(sample_text)

    print("\n" + "="*40)
    print(f"| {'METRİK':<20} | {'DEĞER':<12} |")
    print("="*40)
    print(f"| {'Perplexity Skoru':<20} | {ppl_skoru:<14.2f} |")
    print("-" * 40)

    if ppl_skoru < 40:
        print(">> İSTATİSTİKSEL KARAR: Perplexity çok düşük. Metin muhtemelen YAPAY ZEKA üretimi.")
    else:
        print(">> İSTATİSTİKSEL KARAR: Perplexity yüksek. Metin muhtemelen İNSAN elinden çıkma.")

    token_array_dict = [{"token": token, "count": count} for token, count in token_frekanslari.most_common()]

    burst_skoru = calculate_burstiness(sample_text)
    fk_skoru = calculate_flesch_kincaid(sample_text)
    print(f"| Burstiness (Dalgalanma) | {burst_skoru:<14.2f} |")
    print(f"| Flesch-Kincaid Skoru    | {fk_skoru:<14.2f} |")




[distilbert-base-uncased] tokenizer'ı yükleniyor...

--- Orijinal Metin ---
Quantum computing represents a fundamental shift in how we process information. Unlike classical computers that rely on binary bits, quantum systems utilize qubits, which can exist in multiple states simultaneously due to the principles of superposition and entanglement. This unique characteristic allows quantum computers to solve complex mathematical problems at unprecedented speeds. Furthermore, industries ranging from cryptography to pharmaceuticals are investing heavily in this emerging technology. While significant hardware challenges remain, the potential applications of quantum algorithms are vast and transformative for the future of computational science.

--- Çıkarılan Tokenlar ---
['quantum', 'computing', 'represents', 'a', 'fundamental', 'shift', 'in', 'how', 'we', 'process', 'information', '.', 'unlike', 'classical', 'computers', 'that', 'rely', 'on', 'binary', 'bits', ',', 'quantum', 'systems', 'ut

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



| METRİK               | DEĞER        |
| Perplexity Skoru     | 19.97          |
----------------------------------------
>> İSTATİSTİKSEL KARAR: Perplexity çok düşük. Metin muhtemelen YAPAY ZEKA üretimi.
| Burstiness (Dalgalanma) | 6.50           |
| Flesch-Kincaid Skoru    | 1.50           |

[BİLGİ] Veriler başarıyla dizilere aktarıldı.
Dict Dizisi Eleman  : [{'token': '.', 'count': 5}, {'token': 'quantum', 'count': 4}, {'token': ',', 'count': 4}, {'token': 'in', 'count': 3}, {'token': 'to', 'count': 3}, {'token': 'the', 'count': 3}, {'token': 'of', 'count': 3}, {'token': 'computers', 'count': 2}, {'token': 'and', 'count': 2}, {'token': 'this', 'count': 2}, {'token': 'are', 'count': 2}, {'token': 'computing', 'count': 1}, {'token': 'represents', 'count': 1}, {'token': 'a', 'count': 1}, {'token': 'fundamental', 'count': 1}, {'token': 'shift', 'count': 1}, {'token': 'how', 'count': 1}, {'token': 'we', 'count': 1}, {'token': 'process', 'count': 1}, {'token': 'information', 'count': 1